# Gaze Fixation + Probabilistic Heatmap Action-Type Videos

从 `upper_feature_outcome/gaze_object_features/probabilistic_heatmap_frame_aggregation` 读取 posterior 结果，按动作类别各选一个代表 recording，叠加 gaze heatmap、fixation center 和 YOLO posterior boxes 到完整 world camera 视频。

默认每种动作类别只生成一个完整视频。可以用 `PARTICIPANTS`、`FULL_IDS`、`ACTION_TYPES`、`MAX_VIDEOS` 控制渲染范围。


In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

DATASET_ROOT = Path('/home/julius/datasets/seminar_object_detection')
VIDEO_ROOT = DATASET_ROOT / 'iAssistADL_dataset_export_2026-05-18' / 'world_camera_videos'
UPPER_ROOT = DATASET_ROOT / 'upper_feature_outcome'
GAZE_OBJECT_ROOT = UPPER_ROOT / 'gaze_object_features' / 'probabilistic_heatmap_frame_aggregation'
FIX_SAMPLE_ROOT = UPPER_ROOT / 'gaze_fixation_features' / 'sample_labels'
FIX_SEGMENT_ROOT = UPPER_ROOT / 'gaze_fixation_features' / 'segments'
OUT_ROOT = DATASET_ROOT / 'video_demonstration' / 'gaze_object_probabilistic_heatmap_action_types'

# None means discover everything, then keep one representative recording per action type.
PARTICIPANTS = None  # e.g. ['IA_4_H_03']
FULL_IDS = None      # e.g. ['IA_4_H_03_Teekochen1']
ACTION_TYPES = None  # e.g. ['Brief_schreiben', 'Tee_kochen']
MAX_VIDEOS = None    # e.g. 2 for a quick smoke run
OVERWRITE = False

MAX_OBJECTS = 8
MAX_GAZE_POINTS = 10
HEATMAP_SIGMA = 45
HEATMAP_ALPHA = 0.38

FRAME_SUFFIX = '_gaze_yolo_probabilistic_heatmap_frame.csv'
OBJECT_SUFFIX = '_gaze_yolo_probabilistic_heatmap_frame_objects.csv'
OUTPUT_SUFFIX = '_gaze_fixation_probabilistic_heatmap_full.mp4'

OUT_ROOT.mkdir(parents=True, exist_ok=True)
assert GAZE_OBJECT_ROOT.exists(), f'Missing gaze-object result directory: {GAZE_OBJECT_ROOT}'
GAZE_OBJECT_ROOT


In [ ]:
def action_type_from_full_id(full_id, participant):
    name = str(full_id)
    prefix = f'{participant}_'
    if name.startswith(prefix):
        name = name[len(prefix):]
    name = name.rstrip('0123456789')
    name = name.replace('Teekochen', 'Tee_kochen')
    name = name.replace('Chaos_several_Task_combine', 'Chaos_several_Tasks_combine')
    return name


def discover_actions():
    participant_filter = set(PARTICIPANTS) if PARTICIPANTS else None
    full_id_filter = set(FULL_IDS) if FULL_IDS else None
    action_type_filter = set(ACTION_TYPES) if ACTION_TYPES else None
    rows = []

    for prob_frame_path in sorted(GAZE_OBJECT_ROOT.glob(f'*/**/*{FRAME_SUFFIX}')):
        if prob_frame_path.name.endswith(OBJECT_SUFFIX):
            continue
        participant = prob_frame_path.parent.name
        full_id = prob_frame_path.name.removesuffix(FRAME_SUFFIX)
        action_type = action_type_from_full_id(full_id, participant)
        if participant_filter and participant not in participant_filter:
            continue
        if full_id_filter and full_id not in full_id_filter:
            continue
        if action_type_filter and action_type not in action_type_filter:
            continue

        video_path = VIDEO_ROOT / participant / f'{full_id}_world_camera.mp4'
        fix_sample_path = FIX_SAMPLE_ROOT / participant / f'{full_id}_gaze_fixation_samples.csv'
        fix_segment_path = FIX_SEGMENT_ROOT / participant / f'{full_id}_gaze_fixation_segments.csv'
        prob_object_path = prob_frame_path.with_name(f'{full_id}{OBJECT_SUFFIX}')
        output_path = OUT_ROOT / f'{action_type}{OUTPUT_SUFFIX}'

        rows.append(
            {
                'action_type': action_type,
                'participant': participant,
                'full_id': full_id,
                'video_path': video_path,
                'fix_sample_path': fix_sample_path,
                'fix_segment_path': fix_segment_path,
                'prob_frame_path': prob_frame_path,
                'prob_object_path': prob_object_path,
                'output_path': output_path,
                'has_video': video_path.exists(),
                'has_fix_samples': fix_sample_path.exists(),
                'has_fix_segments': fix_segment_path.exists(),
                'has_prob_objects': prob_object_path.exists(),
                'already_rendered': output_path.exists(),
            }
        )

    actions = pd.DataFrame(rows)
    if actions.empty:
        return actions, actions

    required = ['has_video', 'has_fix_samples', 'has_fix_segments', 'has_prob_objects']
    actions = actions.sort_values(['action_type', 'participant', 'full_id']).reset_index(drop=True)
    renderable = actions[actions[required].all(axis=1)].copy()
    renderable = renderable.drop_duplicates('action_type', keep='first')
    if not OVERWRITE:
        renderable = renderable[~renderable['already_rendered']].copy()
    if MAX_VIDEOS is not None:
        renderable = renderable.head(int(MAX_VIDEOS)).copy()
    return actions, renderable

all_actions, render_queue = discover_actions()
print(f'discovered recordings from: {GAZE_OBJECT_ROOT}')
print(f'all recordings: {len(all_actions)}')
print(f'action types: {all_actions.action_type.nunique() if not all_actions.empty else 0}')
print(f'render queue: {len(render_queue)}')
if not all_actions.empty:
    required = ['has_video', 'has_fix_samples', 'has_fix_segments', 'has_prob_objects']
    print('missing required files:', len(all_actions) - int(all_actions[required].all(axis=1).sum()))
    display(all_actions[['action_type', 'participant', 'full_id', 'has_video', 'has_fix_samples', 'has_fix_segments', 'has_prob_objects', 'already_rendered']].head(30))
    display(render_queue[['action_type', 'participant', 'full_id', 'output_path']])


In [ ]:
def read_video_info(video_path):
    cap = cv2.VideoCapture(str(video_path))
    assert cap.isOpened(), f'Could not open video: {video_path}'
    info = {
        'fps': float(cap.get(cv2.CAP_PROP_FPS)) or 30.0,
        'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        'video_frames': int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    cap.release()
    return info


def clean_frame_index(df, video_frames):
    df = df.copy()
    df['video_frame_idx'] = pd.to_numeric(df['video_frame_idx'], errors='coerce')
    df = df[np.isfinite(df.video_frame_idx)].copy()
    df['video_frame_idx'] = df['video_frame_idx'].astype(int)
    return df[(df.video_frame_idx >= 0) & (df.video_frame_idx < video_frames)].copy()


def load_action_inputs(action):
    info = read_video_info(action.video_path)
    video_frames = info['video_frames']

    fix_samples = pd.read_csv(
        action.fix_sample_path,
        usecols=['video_frame_idx', 'x', 'y', 'fixation_id', 'fixation_candidate'],
    )
    fix_samples = clean_frame_index(fix_samples, video_frames)

    fix_segments = pd.read_csv(action.fix_segment_path)
    if not fix_segments.empty:
        fix_segments['start_video_frame_idx'] = pd.to_numeric(fix_segments['start_video_frame_idx'], errors='coerce')
        fix_segments['end_video_frame_idx'] = pd.to_numeric(fix_segments['end_video_frame_idx'], errors='coerce')
        fix_segments = fix_segments.dropna(subset=['start_video_frame_idx', 'end_video_frame_idx']).copy()
        fix_segments['start_video_frame_idx'] = fix_segments['start_video_frame_idx'].astype(int)
        fix_segments['end_video_frame_idx'] = fix_segments['end_video_frame_idx'].astype(int)

    prob_frame = pd.read_csv(action.prob_frame_path)
    prob_frame = clean_frame_index(prob_frame, video_frames)

    prob_obj = pd.read_csv(action.prob_object_path)
    prob_obj = clean_frame_index(prob_obj, video_frames)

    return info, fix_samples, fix_segments, prob_frame, prob_obj


def build_frame_indexes(fix_samples, prob_frame, prob_obj):
    return {
        'fix_by_frame': {int(k): v for k, v in fix_samples.groupby('video_frame_idx', sort=False)},
        'prob_frame_by_frame': {int(row.video_frame_idx): row.to_dict() for _, row in prob_frame.iterrows()},
        'prob_obj_by_frame': {int(k): v for k, v in prob_obj.groupby('video_frame_idx', sort=False)},
    }


In [ ]:
def posterior_color(prob):
    prob = 0.0 if pd.isna(prob) else float(np.clip(prob, 0, 1))
    return (
        int(255 * (1 - prob)),
        int(255 * (1 - abs(prob - 0.5) * 2)),
        int(255 * prob),
    )


def draw_label(frame, text, x, y, color=(35, 35, 35), scale=0.55, thickness=1):
    font = cv2.FONT_HERSHEY_SIMPLEX
    x, y = int(x), int(y)
    (tw, th), base = cv2.getTextSize(text, font, scale, thickness)
    x = max(0, min(frame.shape[1] - 12, x))
    x2 = min(frame.shape[1] - 1, x + tw + 8)
    y1 = max(0, y - th - base - 6)
    cv2.rectangle(frame, (x, y1), (x2, y + 2), color, -1)
    cv2.putText(frame, text, (x + 4, y - base - 2), font, scale, (255, 255, 255), thickness, cv2.LINE_AA)


def add_heatmap(frame, gaze_rows):
    if gaze_rows is None or gaze_rows.empty:
        return frame
    valid = gaze_rows[np.isfinite(gaze_rows.x) & np.isfinite(gaze_rows.y)]
    if valid.empty:
        return frame
    heat = np.zeros(frame.shape[:2], dtype=np.float32)
    radius = int(round(HEATMAP_SIGMA * 2.4))
    size = radius * 2 + 1
    axis = np.arange(size, dtype=np.float32) - radius
    xx, yy = np.meshgrid(axis, axis)
    kernel = np.exp(-(xx * xx + yy * yy) / (2 * HEATMAP_SIGMA * HEATMAP_SIGMA)).astype(np.float32)
    for _, gaze in valid.iterrows():
        gx, gy = int(round(gaze.x)), int(round(gaze.y))
        x1, x2 = max(0, gx - radius), min(frame.shape[1], gx + radius + 1)
        y1, y2 = max(0, gy - radius), min(frame.shape[0], gy + radius + 1)
        if x1 >= x2 or y1 >= y2:
            continue
        kx1, kx2 = x1 - (gx - radius), size - ((gx + radius + 1) - x2)
        ky1, ky2 = y1 - (gy - radius), size - ((gy + radius + 1) - y2)
        heat[y1:y2, x1:x2] += kernel[ky1:ky2, kx1:kx2]
    if heat.max() <= 0:
        return frame
    heat_norm = np.uint8(np.clip(heat / heat.max() * 255, 0, 255))
    heat_color = cv2.applyColorMap(heat_norm, cv2.COLORMAP_TURBO)
    mask = heat_norm > 5
    blend = cv2.addWeighted(frame, 1 - HEATMAP_ALPHA, heat_color, HEATMAP_ALPHA, 0)
    frame[mask] = blend[mask]
    return frame


In [ ]:
def make_overlay(full_id, fix_segments, indexes):
    fix_by_frame = indexes['fix_by_frame']
    prob_frame_by_frame = indexes['prob_frame_by_frame']
    prob_obj_by_frame = indexes['prob_obj_by_frame']

    def current_fixation(frame_idx):
        if fix_segments.empty:
            return None
        rows = fix_segments[
            (fix_segments.start_video_frame_idx <= frame_idx)
            & (fix_segments.end_video_frame_idx >= frame_idx)
        ]
        if rows.empty:
            return None
        return rows.iloc[0]

    def draw_overlay(frame, frame_idx):
        gaze_rows = fix_by_frame.get(frame_idx)
        frame = add_heatmap(frame, gaze_rows)

        objects = prob_obj_by_frame.get(frame_idx)
        if objects is not None and not objects.empty:
            top = objects.sort_values('object_posterior_probability', ascending=False).head(MAX_OBJECTS)
            for rank, (_, obj) in enumerate(top.iloc[::-1].iterrows()):
                prob = float(obj.object_posterior_probability)
                color = posterior_color(prob)
                x1, y1, x2, y2 = [int(round(obj[col])) for col in ['x1', 'y1', 'x2', 'y2']]
                is_best = rank == len(top) - 1
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 4 if is_best else 2)
                if is_best or prob >= 0.15:
                    draw_label(frame, f'cls {int(obj.class_id)} p={prob:.2f}', x1, max(22, y1 - 4), color=color, scale=0.55)

        if gaze_rows is not None and not gaze_rows.empty:
            valid = gaze_rows[np.isfinite(gaze_rows.x) & np.isfinite(gaze_rows.y)]
            for _, gaze in valid.tail(MAX_GAZE_POINTS).iterrows():
                gx, gy = int(round(gaze.x)), int(round(gaze.y))
                cv2.circle(frame, (gx, gy), 5, (255, 255, 255), -1)
                cv2.circle(frame, (gx, gy), 9, (0, 0, 0), 2)

        fixation = current_fixation(frame_idx)
        if fixation is not None:
            cx, cy = int(round(fixation.center_x)), int(round(fixation.center_y))
            cv2.circle(frame, (cx, cy), 18, (0, 255, 255), 3)
            cv2.drawMarker(frame, (cx, cy), (0, 255, 255), markerType=cv2.MARKER_CROSS, markerSize=28, thickness=2)
            draw_label(frame, f'fix {int(fixation.fixation_id)} {float(fixation.duration_ms):.0f}ms', cx + 20, max(26, cy - 16), color=(0, 120, 120), scale=0.55)

        summary = prob_frame_by_frame.get(frame_idx, {})
        best_class = summary.get('best_class', 'NA')
        best_prob = summary.get('best_object_probability', np.nan)
        entropy = summary.get('object_probability_normalized_entropy', np.nan)
        detections = summary.get('yolo_detection_count', 0)
        title = f'{full_id} | frame {frame_idx} | prob heatmap best={best_class} p={0 if pd.isna(best_prob) else float(best_prob):.2f} ent={0 if pd.isna(entropy) else float(entropy):.2f} det={int(detections) if pd.notna(detections) else 0}'
        draw_label(frame, title, 12, 32, color=(35, 35, 35), scale=0.62)
        draw_label(frame, 'gaze heatmap + fixation center + YOLO posterior boxes', 12, 62, color=(35, 35, 35), scale=0.52)
        return frame

    return draw_overlay


def open_writer(path, fps, size):
    path.parent.mkdir(parents=True, exist_ok=True)
    for codec in ['mp4v', 'avc1', 'XVID']:
        writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*codec), fps, size)
        if writer.isOpened():
            return writer, codec
        writer.release()
    raise RuntimeError(f'Could not open writer: {path}')


In [ ]:
def render_full_action(action):
    info, fix_samples, fix_segments, prob_frame, prob_obj = load_action_inputs(action)
    indexes = build_frame_indexes(fix_samples, prob_frame, prob_obj)
    draw_overlay = make_overlay(f'{action.action_type} | {action.full_id}', fix_segments, indexes)

    cap = cv2.VideoCapture(str(action.video_path))
    assert cap.isOpened(), f'Could not open video: {action.video_path}'

    writer, codec = open_writer(action.output_path, info['fps'], (info['width'], info['height']))
    written = 0
    progress = tqdm(
        total=info['video_frames'],
        desc=f'Rendering {action.action_type}',
        unit='frame',
        leave=False,
    )

    try:
        for frame_idx in range(info['video_frames']):
            ok, frame = cap.read()
            if not ok:
                print('stopped early:', action.full_id, 'at frame', frame_idx)
                break
            writer.write(draw_overlay(frame, frame_idx))
            written += 1
            progress.update(1)
    finally:
        progress.close()
        writer.release()
        cap.release()

    if written == 0:
        raise RuntimeError(f'No frames were written for {action.action_type}')

    size_mb = action.output_path.stat().st_size / (1024 * 1024)
    return {
        'action_type': action.action_type,
        'participant': action.participant,
        'full_id': action.full_id,
        'output_path': action.output_path,
        'frames': written,
        'fps': info['fps'],
        'codec': codec,
        'size_mb': size_mb,
    }


render_results = []
for action in tqdm(list(render_queue.itertuples(index=False)), desc='Actions', unit='video'):
    print('rendering:', action.action_type, 'from', action.full_id)
    result = render_full_action(action)
    render_results.append(result)
    print(f"saved: {result['output_path']} ({result['frames']} frames, {result['size_mb']:.1f} MB, codec={result['codec']})")

render_results = pd.DataFrame(render_results)
render_results
